# Handling Missing Data

Real datasets always have holes: a survey question left blank, a sensor that failed, a form field nobody filled. Most ML models **cannot work with missing values** (they crash or give wrong results), so handling them is a core skill and a favourite interview topic.

**Contents**
1. Why data goes missing (3 types)
2. How pandas represents missing data
3. Fake missing values (`?`, `-`, `0`, `-999`)
4. Detecting missing data
5. How NaN behaves in calculations and filters
6. Dropping: `dropna`
7. Filling: `fillna` (constant, mean, median, mode)
8. Mean vs median: which one?
9. Forward fill, backward fill, interpolation
10. Group-wise filling
11. Filling from another column
12. Missing indicator columns
13. What imputation does to your data
14. Imputation without data leakage (train/test)
15. Decision guide
16. Pandas 3.x notes
17. Mini AI/ML example: a full missing-data pipeline
18. Common mistakes
19. Interview questions
20. Practice questions with solutions
21. Cheat sheet

In [2]:
import numpy as np
import pandas as pd
import warnings

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. Why data goes missing (3 types)

Before choosing a method, ask: **why** is the value missing? The reason decides what is safe to do.

| Type | Meaning | Example | Safe to drop/fill? |
|---|---|---|---|
| **MCAR** (Missing Completely At Random) | Missing for no reason, unrelated to any data | A few survey forms got lost in the mail | Mostly yes |
| **MAR** (Missing At Random) | Missing depends on **other columns you can see** | Students without internet skip the "online hours" question | Yes, if you use the other columns (e.g. group-wise fill) |
| **MNAR** (Missing Not At Random) | Missing depends on **the missing value itself** | High-income people refuse to tell their income | Dangerous, filling can bias the result. Create an indicator and think carefully |

Interview point: **the fact that a value is missing can itself be information** (for example, a customer who left the "phone number" empty).

---
# 2. How pandas represents missing data

| Symbol | Where you see it |
|---|---|
| `np.nan` (NaN) | Missing number (it is a float, so int columns with NaN become float) |
| `None` | Python missing value, pandas converts it to NaN in most columns |
| `pd.NA` | The newer "nullable" missing value (used by `Int64`, `boolean`, `string` dtypes) |
| `NaT` | Missing **datetime** |

All of them are detected by `isna()`.

In [3]:
s = pd.Series([1, None, 3])
print(s)                                   # None became NaN, dtype became float64
print(s.dtype)

0    1.0
1    NaN
2    3.0
dtype: float64
float64


In [4]:
# Integer column with missing values: the nullable "Int64" keeps integers
s_int = pd.Series([1, None, 3], dtype="Int64")
print(s_int)
print(s_int.dtype)

0       1
1    <NA>
2       3
dtype: Int64
Int64


In [5]:
# Missing datetime is NaT
d = pd.to_datetime(pd.Series(["2026-01-05", None, "2026-03-10"]))
print(d)
print(d.isna().tolist())

0   2026-01-05
1          NaT
2   2026-03-10
dtype: datetime64[us]
[False, True, False]


In [6]:
# NaN is not equal to anything, not even to itself
print(np.nan == np.nan)
print(np.nan is np.nan)
print(pd.isna(np.nan), pd.isna(None), pd.isna(pd.NA), pd.isna(pd.NaT))

False
True
True True True True


---
# 3. Fake missing values (`?`, `-`, `0`, `-999`)

The worst missing values are the ones **pandas cannot see**. People type `?`, `N/A`, `-`, `unknown`, an empty string, `0` for age, or `-999` as a code. `isna()` shows zero missing, but the data is still broken.

How to find them: use `value_counts()` and `describe()` (chapter 04), look for suspicious values.

In [7]:
messy = pd.DataFrame({
    "age":    ["21", "?", "23", "N/A", "25"],
    "salary": [30000, -999, 45000, 38000, -999],
    "city":   ["Indore", "", "Delhi", "unknown", "Pune"],
})

print(messy.isna().sum())          # shows 0 missing everywhere!
print()
print(messy["city"].value_counts())
print(messy["salary"].describe()[["min", "max"]])

age       0
salary    0
city      0
dtype: int64

city
Indore     1
           1
Delhi      1
unknown    1
Pune       1
Name: count, dtype: int64
min     -999.0
max    45000.0
Name: salary, dtype: float64


In [8]:
# Convert all fake markers into real NaN
clean = messy.replace(["?", "N/A", "", "unknown", -999], np.nan)
clean["age"] = pd.to_numeric(clean["age"])         # text numbers -> real numbers

print(clean)
print()
print(clean.isna().sum())
print(clean.dtypes)

    age   salary    city
0  21.0  30000.0  Indore
1   NaN      NaN     NaN
2  23.0  45000.0   Delhi
3   NaN  38000.0     NaN
4  25.0      NaN    Pune

age       2
salary    2
city      2
dtype: int64
age       float64
salary    float64
city          str
dtype: object


Best habit: tell pandas at **load time** with `pd.read_csv(..., na_values=["?", "N/A", "-", "unknown"])` (chapter 03).

---
# 4. Detecting missing data

Our practice table: small, with holes in every type of column.

In [9]:
people = pd.DataFrame({
    "name":   ["Ravi", "Asha", "Karan", "Neha", "Amit", "Sara", "Dev", "Pooja"],
    "age":    [21, np.nan, 23, 22, np.nan, 24, 21, 25],
    "city":   ["Indore", "Bhopal", None, "Indore", "Pune", None, "Bhopal", "Indore"],
    "income": [30, 32, np.nan, 31, 500, np.nan, 33, 29],      # monthly income in thousands (one huge value)
    "score":  [85, 92, 58, np.nan, 64, 97, np.nan, 81],
})
people

,name,age,city,income,score
0,Ravi,21.0,Indore,30.0,85.0
1,Asha,NaN,Bhopal,32.0,92.0
2,Karan,23.0,NaN,NaN,58.0
3,Neha,22.0,Indore,31.0,NaN
4,Amit,NaN,Pune,500.0,64.0
5,Sara,24.0,NaN,NaN,97.0
6,Dev,21.0,Bhopal,33.0,NaN
7,Pooja,25.0,Indore,29.0,81.0


In [10]:
people.isna()          # True where missing

,name,age,city,income,score
0,False,False,False,False,False
1,False,True,False,False,False
2,False,False,True,True,False
3,False,False,False,False,True
4,False,True,False,False,False
5,False,False,True,True,False
6,False,False,False,False,True
7,False,False,False,False,False


In [11]:
people.isna().sum()                              # count per column

name      0
age       2
city      2
income    2
score     2
dtype: int64

In [12]:
(people.isna().mean() * 100).round(1)            # percentage per column

name       0.0
age       25.0
city      25.0
income    25.0
score     25.0
dtype: float64

In [13]:
print("total missing cells:", people.isna().sum().sum())
print("rows with any missing:", people.isna().any(axis=1).sum())
print("rows with ALL missing:", people.isna().all(axis=1).sum())

total missing cells: 8
rows with any missing: 6
rows with ALL missing: 0


In [14]:
# Which rows have missing values?
people[people.isna().any(axis=1)]

,name,age,city,income,score
1,Asha,NaN,Bhopal,32.0,92.0
2,Karan,23.0,NaN,NaN,58.0
3,Neha,22.0,Indore,31.0,NaN
4,Amit,NaN,Pune,500.0,64.0
5,Sara,24.0,NaN,NaN,97.0
6,Dev,21.0,Bhopal,33.0,NaN


In [15]:
# How many missing values per row?
people.isna().sum(axis=1)

0    0
1    1
2    2
3    1
4    1
5    2
6    1
7    0
dtype: int64

In [16]:
# A compact missing-data report you can reuse
def missing_report(df):
    rep = pd.DataFrame({
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "dtype": df.dtypes.astype(str),
    })
    return rep[rep["missing"] > 0].sort_values("missing", ascending=False)

missing_report(people)

,missing,missing_%,dtype
age,2,25.0,float64
city,2,25.0,str
income,2,25.0,float64
score,2,25.0,float64


## Is the missingness random? (looking for MAR)

Check whether missing values **depend on another column**. If the missing rate differs a lot between groups, the data is not missing at random.

In [17]:
# Missing rate of "score" for each city
people.assign(score_missing=people["score"].isna()).groupby("city")["score_missing"].mean()

city
Bhopal    0.500000
Indore    0.333333
Pune      0.000000
Name: score_missing, dtype: float64

With real, bigger data this pattern check (missing rate per group) tells you whether a simple global fill is safe or whether group-wise filling is better.

---
# 5. How NaN behaves in calculations and filters

## 5.1 Maths and statistics

In [18]:
s = pd.Series([10, np.nan, 30])

print("s + 5      :", (s + 5).tolist())              # NaN stays NaN
print("sum        :", s.sum())                         # NaN is skipped by default
print("sum no skip:", s.sum(skipna=False))             # nan
print("mean       :", s.mean())                        # (10+30)/2, NaN is not counted
print("count      :", s.count())                       # counts only non-missing
print("len        :", len(s))                          # counts all rows

s + 5      : [15.0, nan, 35.0]
sum        : 40.0
sum no skip: nan
mean       : 20.0
count      : 2
len        : 3


**Careful:** `mean()` divides by the number of **non-missing** values, not by the total rows.

## 5.2 Filters silently drop NaN rows (a classic trap)

In [19]:
print((people["income"] > 30).tolist())

[False, True, False, True, True, False, True, False]


In [20]:
print("income > 30  :", (people["income"] > 30).sum(), "rows")
print("income <= 30 :", (people["income"] <= 30).sum(), "rows")
print("total rows   :", len(people))
# 8 rows but the two counts add up to less: NaN rows are in neither group!

income > 30  : 4 rows
income <= 30 : 2 rows
total rows   : 8


In [21]:
# NOT-condition includes NaN rows (because NaN > 30 is False, so ~False is True)
print((~(people["income"] > 30)).sum())

4


So **`~(x > 30)` is not the same as `x <= 30`** when NaN exists. Always handle missing values before filtering, or write the condition explicitly: `(x <= 30) | x.isna()`.

## 5.3 groupby and value_counts ignore NaN by default

In [22]:
print(people.groupby("city").size())                    # rows with missing city disappear
print()
print(people.groupby("city", dropna=False).size())      # keep them as their own group
print()
print(people["city"].value_counts(dropna=False))

city
Bhopal    2
Indore    3
Pune      1
dtype: int64

city
Bhopal    2
Indore    3
Pune      1
NaN       2
dtype: int64

city
Indore    3
Bhopal    2
NaN       2
Pune      1
Name: count, dtype: int64


---
# 6. Dropping: `dropna`

Simplest solution: remove rows (or columns) with missing values. But **you lose information**.

In [23]:
print("original:", people.shape)
print("dropna() :", people.dropna().shape)         # row dropped if ANY value is missing

original: (8, 5)
dropna() : (2, 5)


In [24]:
# only look at selected columns
people.dropna(subset=["score"])

,name,age,city,income,score
0,Ravi,21.0,Indore,30.0,85.0
1,Asha,NaN,Bhopal,32.0,92.0
2,Karan,23.0,NaN,NaN,58.0
4,Amit,NaN,Pune,500.0,64.0
5,Sara,24.0,NaN,NaN,97.0
7,Pooja,25.0,Indore,29.0,81.0


In [25]:
# keep rows having at least 4 non-missing values
people.dropna(thresh=4)

,name,age,city,income,score
0,Ravi,21.0,Indore,30.0,85.0
1,Asha,NaN,Bhopal,32.0,92.0
3,Neha,22.0,Indore,31.0,NaN
4,Amit,NaN,Pune,500.0,64.0
6,Dev,21.0,Bhopal,33.0,NaN
7,Pooja,25.0,Indore,29.0,81.0


In [26]:
# how="all": drop only rows where EVERYTHING is missing
people.dropna(how="all").shape

(8, 5)

In [27]:
# Drop COLUMNS (axis=1) with too many missing values
big_gap = people.assign(garbage=[np.nan] * 7 + [1])

limit = 0.4                                          # allow at most 40% missing
keep = big_gap.columns[big_gap.isna().mean() <= limit]
big_gap[keep].columns.tolist()

['name', 'age', 'city', 'income', 'score']

### When is dropping OK?

| Situation | Drop? |
|---|---|
| Missing rows are a tiny part (less than 5%) and random (MCAR) | Yes |
| The **target** (what you predict) is missing | Yes, drop those rows (you cannot learn from them) |
| A column is mostly empty (more than 40-60%) and not important | Drop the column |
| Small dataset | Avoid, every row is precious |
| Missingness has a pattern (MAR/MNAR) | No, dropping biases the data |

Always measure what you lose: `1 - len(df.dropna()) / len(df)`.

In [28]:
loss = 1 - len(people.dropna()) / len(people)
print(f"dropna() would remove {loss:.0%} of the data")

dropna() would remove 75% of the data


---
# 7. Filling: `fillna`

## 7.1 Fill with a constant

In [29]:
people["score"].fillna(0)

0    85.0
1    92.0
2    58.0
3     0.0
4    64.0
5    97.0
6     0.0
7    81.0
Name: score, dtype: float64

In [30]:
# different value for each column using a dict
people.fillna({"age": 0, "city": "Unknown", "income": -1}).head(4)

,name,age,city,income,score
0,Ravi,21.0,Indore,30.0,85.0
1,Asha,0.0,Bhopal,32.0,92.0
2,Karan,23.0,Unknown,-1.0,58.0
3,Neha,22.0,Indore,31.0,NaN


Use a constant when the value has a real meaning ("Unknown" for text, 0 for "no purchases"). Do **not** fill ages or scores with 0, it creates fake data.

## 7.2 Fill numbers with mean, median or mode

In [31]:
age_mean = people["age"].mean()
score_median = people["score"].median()
print("age mean     :", round(age_mean, 2))
print("score median :", score_median)

d = people.copy()
d["age"] = d["age"].fillna(age_mean)
d["score"] = d["score"].fillna(score_median)
d[["name", "age", "score"]]

age mean     : 22.67
score median : 83.0


,name,age,score
0,Ravi,21.000000,85.0
1,Asha,22.666667,92.0
2,Karan,23.000000,58.0
3,Neha,22.000000,83.0
4,Amit,22.666667,64.0
5,Sara,24.000000,97.0
6,Dev,21.000000,83.0
7,Pooja,25.000000,81.0


## 7.3 Fill text/categories with the mode (most frequent value)

In [32]:
print(people["city"].mode())            # mode returns a Series (there can be ties)
most_common = people["city"].mode()[0]  # take the first one
print("most common city:", most_common)

people["city"].fillna(most_common)

0    Indore
Name: city, dtype: str
most common city: Indore


0    Indore
1    Bhopal
2    Indore
3    Indore
4      Pune
5    Indore
6    Bhopal
7    Indore
Name: city, dtype: str

In [33]:
# Or admit that it is unknown (often better than guessing)
people["city"].fillna("Unknown")

0     Indore
1     Bhopal
2    Unknown
3     Indore
4       Pune
5    Unknown
6     Bhopal
7     Indore
Name: city, dtype: str

---
# 8. Mean vs median: which one?

- **Mean** is pulled by outliers.
- **Median** is the middle value, robust to outliers.

Look at `income`: one person earns 500 and everyone else about 30.

In [34]:
print("incomes:", people["income"].dropna().tolist())
print("mean  :", round(people["income"].mean(), 1))
print("median:", people["income"].median())

incomes: [30.0, 32.0, 31.0, 500.0, 33.0, 29.0]
mean  : 109.2
median: 31.5


If we filled the missing incomes with the **mean** (about 109), two people would suddenly get an income that **nobody in the typical group** has. The median (32) is much more sensible.

| Data | Use |
|---|---|
| Roughly symmetric, no outliers | Mean |
| Skewed or has outliers (income, price, house size) | **Median** |
| Categories / text | Mode or "Unknown" |
| Time ordered data | Forward fill / interpolation |

When in doubt for numbers: **median**.

---
# 9. Forward fill, backward fill, interpolation

For **ordered** data (time series, sensor readings, stock prices) the neighbours carry information.

In [35]:
days = pd.date_range("2026-01-01", periods=8)
temp = pd.Series([20, np.nan, np.nan, 26, np.nan, 28, np.nan, np.nan], index=days, name="temp")
temp

2026-01-01    20.0
2026-01-02     NaN
2026-01-03     NaN
2026-01-04    26.0
2026-01-05     NaN
2026-01-06    28.0
2026-01-07     NaN
2026-01-08     NaN
Freq: D, Name: temp, dtype: float64

In [36]:
print("ffill:", temp.ffill().tolist())      # copy the last known value forward
print("bfill:", temp.bfill().tolist())      # copy the next known value backward

ffill: [20.0, 20.0, 20.0, 26.0, 26.0, 28.0, 28.0, 28.0]
bfill: [20.0, 26.0, 26.0, 26.0, 28.0, 28.0, nan, nan]


In [37]:
# limit: do not carry the value more than N steps
print(temp.ffill(limit=1).tolist())

[20.0, 20.0, nan, 26.0, 26.0, 28.0, 28.0, nan]


In [38]:
# interpolate: draw a straight line between the known points
print(temp.interpolate().tolist())

[20.0, 22.0, 24.0, 26.0, 27.0, 28.0, 28.0, 28.0]


In [39]:
# method="time" respects the real gaps between dates
uneven = pd.Series([10, np.nan, np.nan, 40],
                   index=pd.to_datetime(["2026-01-01", "2026-01-02", "2026-01-03", "2026-01-10"]))
print("linear:", uneven.interpolate().tolist())
print("time  :", uneven.interpolate(method="time").round(1).tolist())

linear: [10.0, 20.0, 30.0, 40.0]
time  : [10.0, 13.3, 16.7, 40.0]


- `ffill` is natural for "last known state" (price, status).
- `interpolate` is natural for smooth values (temperature).
- Both only make sense if the rows are **in a meaningful order** (sort first!).

---
# 10. Group-wise filling

A global median ignores context. A student from a big city and one from a small town may differ. Fill each missing value using the **median of its own group**.

In [40]:
people["score_group_fill"] = people["score"].fillna(
    people.groupby("city")["score"].transform("median")
)
people[["name", "city", "score", "score_group_fill"]]

,name,city,score,score_group_fill
0,Ravi,Indore,85.0,85.0
1,Asha,Bhopal,92.0,92.0
2,Karan,NaN,58.0,58.0
3,Neha,Indore,NaN,83.0
4,Amit,Pune,64.0,64.0
5,Sara,NaN,97.0,97.0
6,Dev,Bhopal,NaN,92.0
7,Pooja,Indore,81.0,81.0


- `transform("median")` gives every row the median of **its city** (full groupby in chapter 10).
- Rows whose group has no known value (city itself missing) stay NaN. Fill with the global median afterwards:

In [41]:
global_median = people["score"].median()
people["score_group_fill"] = people["score_group_fill"].fillna(global_median)
people[["name", "city", "score", "score_group_fill"]]

,name,city,score,score_group_fill
0,Ravi,Indore,85.0,85.0
1,Asha,Bhopal,92.0,92.0
2,Karan,NaN,58.0,58.0
3,Neha,Indore,NaN,83.0
4,Amit,Pune,64.0,64.0
5,Sara,NaN,97.0,97.0
6,Dev,Bhopal,NaN,92.0
7,Pooja,Indore,81.0,81.0


---
# 11. Filling from another column

In [42]:
contacts = pd.DataFrame({
    "mobile": ["98xxx", None, "97xxx", None],
    "home":   ["07xxx", "08xxx", None,  None],
})

# use "mobile", and where it is missing use "home"
contacts["best_contact"] = contacts["mobile"].fillna(contacts["home"])
contacts

,mobile,home,best_contact
0,98xxx,07xxx,98xxx
1,NaN,08xxx,08xxx
2,97xxx,NaN,97xxx
3,NaN,NaN,NaN


In [43]:
# combine_first does the same for whole Series/DataFrames
contacts["mobile"].combine_first(contacts["home"]).tolist()

['98xxx', '08xxx', '97xxx', nan]

---
# 12. Missing indicator columns

Sometimes **"was it missing?"** is useful information for a model. Before filling, save that fact in a new 0/1 column.

In [44]:
d = people[["name", "age", "score"]].copy()

d["age_missing"] = d["age"].isna().astype(int)
d["age"] = d["age"].fillna(d["age"].median())

d

,name,age,score,age_missing
0,Ravi,21.0,85.0,0
1,Asha,22.5,92.0,1
2,Karan,23.0,58.0,0
3,Neha,22.0,NaN,0
4,Amit,22.5,64.0,1
5,Sara,24.0,97.0,0
6,Dev,21.0,NaN,0
7,Pooja,25.0,81.0,0


This is exactly what `SimpleImputer(add_indicator=True)` does in scikit-learn. It is especially important for MNAR data.

---
# 13. What imputation does to your data

Filling is not free. It changes your data. Let us measure it on a larger dataset.

In [45]:
rng = np.random.default_rng(3)
n = 300

big = pd.DataFrame({
    "internet": rng.choice([0, 1], n, p=[0.25, 0.75]),
    "hours": rng.normal(5, 1.6, n).clip(0.5, 10).round(1),
})
big["attendance"] = (55 + 4 * big["hours"] + 8 * big["internet"] + rng.normal(0, 5, n)).clip(30, 100).round(0)
big["score"] = (10 + 5 * big["hours"] + 0.3 * big["attendance"] + rng.normal(0, 5, n)).clip(0, 100).round(0)

true_attendance = big["attendance"].copy()      # we keep the real values only to judge the fills later

# make attendance missing, MORE often when there is no internet (this is MAR)
p_missing = np.where(big["internet"] == 0, 0.35, 0.05)
big.loc[rng.random(n) < p_missing, "attendance"] = np.nan

print(missing_report(big))

            missing  missing_%    dtype
attendance       39       13.0  float64


In [46]:
# 1. Is it random? NO: it depends on internet (MAR)
big.assign(att_missing=big["attendance"].isna()).groupby("internet")["att_missing"].mean().round(3)

internet
0    0.364
1    0.064
Name: att_missing, dtype: float64

In [47]:
# 2. Compare four fill strategies
was_missing = big["attendance"].isna()
known = big["attendance"].dropna()

fill_mean = big["attendance"].fillna(big["attendance"].mean())
fill_group = big["attendance"].fillna(big.groupby("internet")["attendance"].transform("median"))

# model-based: predict attendance from hours + internet with a straight-line model (least squares)
A = np.column_stack([np.ones(len(big)), big["hours"], big["internet"]])
coef, *_ = np.linalg.lstsq(A[~was_missing], known.to_numpy(), rcond=None)
predicted = pd.Series(A @ coef, index=big.index)
fill_model = big["attendance"].fillna(predicted)

fills = {"global mean fill": fill_mean, "group-wise median fill": fill_group, "model-based fill": fill_model}

summary = pd.DataFrame({
    "std": [known.std()] + [f.std() for f in fills.values()],
    "corr_with_score": [big["attendance"].corr(big["score"])] + [f.corr(big["score"]) for f in fills.values()],
    "avg_error_on_filled_rows": [np.nan] + [(f[was_missing] - true_attendance[was_missing]).abs().mean() for f in fills.values()],
}, index=["only known values (reference)"] + list(fills.keys())).round(3)
summary

,std,corr_with_score,avg_error_on_filled_rows
only known values (reference),8.701,0.793,NaN
global mean fill,8.113,0.738,8.358
group-wise median fill,8.363,0.715,5.846
model-based fill,8.624,0.780,4.685


How to read this table (the real attendance of the filled rows is known here, because we created the data):

- **`std`**: the global mean fill **shrinks the spread** (many identical values were added). Better fills keep it closer to the reference.
- **`corr_with_score`**: the mean fill **weakens** the link between attendance and score. Group-wise fill by `internet` alone does not repair it, because the filled values still carry no information about `hours`. The model-based fill, which also uses `hours`, keeps the relationship close to the reference.
- **`avg_error_on_filled_rows`**: how far the filled values are from the true ones. It drops step by step: mean -> group-wise -> model-based.

Takeaway: a simple fill is fast but not free. The more of the **other columns** a fill uses, the better it keeps the structure of your data. For important columns, consider group-wise, KNN or model-based imputation (`KNNImputer`, `IterativeImputer`).

---
# 14. Imputation without data leakage (train/test)

**Wrong:** compute the mean from the **whole dataset**, fill, and then split into train and test. The test set has influenced the fill value, so information from the test data "leaked" into training.

**Right:** compute the fill statistics **only from the training set**, then apply the **same numbers** to the test set.

In [48]:
data = big.sample(frac=1, random_state=42).reset_index(drop=True)
split = int(0.8 * len(data))
train, test = data.iloc[:split].copy(), data.iloc[split:].copy()

# learn the value from TRAIN only
train_median = train["attendance"].median()
print("median learned from train:", train_median)

# apply the same value to both
train["attendance"] = train["attendance"].fillna(train_median)
test["attendance"] = test["attendance"].fillna(train_median)

print("train missing:", train["attendance"].isna().sum(), "| test missing:", test["attendance"].isna().sum())

median learned from train: 83.0
train missing: 0 | test missing: 0


In real projects scikit-learn does this for you safely:

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)     # fit = learn the median from train
X_test = imputer.transform(X_test)           # transform only, never fit on test
```

Other imputers to know: `KNNImputer` (fill using similar rows), `IterativeImputer` (predict each column from the others).

---
# 15. Decision guide

| Situation | What to do |
|---|---|
| Missing in **target** column | Drop those rows |
| Less than 5% missing, random | Drop rows or fill simply |
| Numeric, no outliers | Fill with mean |
| Numeric, skewed or outliers | Fill with **median** |
| Categorical | Mode or `"Unknown"` |
| Missing depends on another column (MAR) | Group-wise fill |
| Time series | `ffill`, `bfill`, `interpolate` |
| "Missing" itself may mean something | Add an **indicator column** first |
| More than 40-60% missing | Consider dropping the column |
| Important column, lots of missing | KNN / model-based imputation |
| Using train/test | Learn the fill value from **train only** |
| Fake markers (`?`, `-999`) | Convert to NaN **first** |

Order of work: **(1) convert fake missing to NaN -> (2) detect and understand the pattern -> (3) drop what must go -> (4) add indicators -> (5) fill -> (6) verify with `isna().sum()`.**

---
# 16. Pandas 3.x notes

- `fillna(method="ffill")` is **removed**. Use `df.ffill()` / `df.bfill()`.
- Chained inplace like `df["col"].fillna(0, inplace=True)` **does not change `df`** under Copy-on-Write and gives a warning. Always assign back: `df["col"] = df["col"].fillna(0)`.
- Text columns use the `str` dtype, and missing text is still shown as `NaN`.
- Ints with missing values become float64 unless you use the nullable `"Int64"` dtype.

In [49]:
tmp = people[["name", "score"]].copy()

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    tmp["score"].fillna(0, inplace=True)          # WRONG way
    print("warning:", w[0].category.__name__ if w else "none")

print("still missing after wrong way:", tmp["score"].isna().sum())

tmp["score"] = tmp["score"].fillna(0)             # RIGHT way
print("still missing after right way:", tmp["score"].isna().sum())

still missing after wrong way: 2
still missing after right way: 0


---
# 17. Mini AI/ML example: a full missing-data pipeline

Task: prepare `raw` for a model that predicts `price_lakh`. The table has real-looking problems: fake markers, missing numbers, missing text, missing target.

In [50]:
raw = pd.DataFrame({
    "area":      [850, 1200, "?", 700, 2000, 1100, np.nan, 1500],
    "bedrooms":  [2, 3, 3, np.nan, 4, 2, 3, np.nan],
    "age_years": [10, 5, np.nan, 15, 2, 8, np.nan, 12],
    "city":      ["Indore", "Bhopal", "Indore", None, "Pune", "Bhopal", "Indore", "Pune"],
    "price_lakh": [45, 70, 85, 30, 120, 60, 55, np.nan],
})
print(raw)
print()
print(raw.dtypes)

   area  bedrooms  age_years    city  price_lakh
0   850       2.0       10.0  Indore        45.0
1  1200       3.0        5.0  Bhopal        70.0
2     ?       3.0        NaN  Indore        85.0
3   700       NaN       15.0     NaN        30.0
4  2000       4.0        2.0    Pune       120.0
5  1100       2.0        8.0  Bhopal        60.0
6   NaN       3.0        NaN  Indore        55.0
7  1500       NaN       12.0    Pune         NaN

area           object
bedrooms      float64
age_years     float64
city              str
price_lakh    float64
dtype: object


In [51]:
# Step 1: fake missing -> real NaN, fix dtypes
df = raw.replace("?", np.nan)
df["area"] = pd.to_numeric(df["area"])

# Step 2: detect
print(missing_report(df))

            missing  missing_%    dtype
area              2       25.0  float64
bedrooms          2       25.0  float64
age_years         2       25.0  float64
city              1       12.5      str
price_lakh        1       12.5  float64


In [52]:
# Step 3: drop rows where the TARGET is missing
df = df.dropna(subset=["price_lakh"]).reset_index(drop=True)
print("rows left:", len(df))

rows left: 7


In [53]:
# Step 4: indicator for a column where missing may carry meaning
df["age_missing"] = df["age_years"].isna().astype(int)

In [54]:
# Step 5: fill
df["area"] = df["area"].fillna(df.groupby("city")["area"].transform("median"))   # group-wise
df["area"] = df["area"].fillna(df["area"].median())                              # fallback
df["bedrooms"] = df["bedrooms"].fillna(df["bedrooms"].mode()[0])                 # mode (count-like)
df["age_years"] = df["age_years"].fillna(df["age_years"].median())               # median
df["city"] = df["city"].fillna("Unknown")                                        # text

# Step 6: verify
print("missing cells left:", df.isna().sum().sum())
df

missing cells left: 0


,area,bedrooms,age_years,city,price_lakh,age_missing
0,850.0,2.0,10.0,Indore,45.0,0
1,1200.0,3.0,5.0,Bhopal,70.0,0
2,850.0,3.0,8.0,Indore,85.0,1
3,700.0,3.0,15.0,Unknown,30.0,0
4,2000.0,4.0,2.0,Pune,120.0,0
5,1100.0,2.0,8.0,Bhopal,60.0,0
6,850.0,3.0,8.0,Indore,55.0,1


Everything is numeric or a clean category now, so the table is ready for encoding (`get_dummies`) and then a model. (In a real project, fill statistics are learned from the train split only, as in section 14.)

---
# 18. Common mistakes

| Mistake | Fix |
|---|---|
| Trusting `isna().sum() == 0` | Look for fake markers (`?`, `-999`, `""`, `0`) with `value_counts` / `describe` |
| `df["a"] == np.nan` | `df["a"].isna()` |
| `dropna()` on the whole table without checking the loss | Measure first, use `subset=` |
| Filling everything with 0 | 0 is a real value, it creates fake data |
| Mean fill on skewed data | Use the median |
| `df["a"].fillna(x, inplace=True)` | `df["a"] = df["a"].fillna(x)` |
| `fillna(method="ffill")` | `df.ffill()` |
| Filling before splitting train/test using full-data statistics | Learn from train only (data leakage) |
| `~(x > 5)` expecting `x <= 5` | NaN rows are included in the `~` version |
| Forgetting groupby drops NaN keys | `groupby(..., dropna=False)` |
| Using `ffill` on unsorted data | Sort by time first |
| Filling the target column | Drop rows with missing target |
| Ignoring that "missing" may be informative | Add an indicator column |

---
# 19. Interview questions

**Q1. How do you handle missing values?**
First detect and understand them (`isna().sum()`, pattern per group, fake markers). Then drop rows/columns if the loss is small and random, or fill: mean/median for numbers, mode or "Unknown" for categories, group-wise fill, `ffill`/`interpolate` for time series, or model-based imputation (KNN/Iterative). Add an indicator column when missingness is informative.

**Q2. Mean or median for imputation?**
Median when the data is skewed or has outliers, mean when it is roughly symmetric.

**Q3. What are MCAR, MAR and MNAR?**
Missing completely at random (no pattern), missing at random (depends on other observed columns), missing not at random (depends on the missing value itself). MNAR is the risky one.

**Q4. What is data leakage in imputation?**
Computing fill values from the whole dataset (including test data) before splitting. Fit the imputer on train only, then transform train and test.

**Q5. `isna()` vs `== np.nan`?**
NaN is not equal to itself, so `== np.nan` is always False. `isna()` is the correct check.

**Q6. Difference between NaN, None, pd.NA, NaT?**
NaN is a float missing value, None is Python's null (usually converted to NaN), `pd.NA` is the nullable-dtype missing value, NaT is a missing datetime.

**Q7. How does pandas treat NaN in `sum` and `mean`?**
By default it skips them (`skipna=True`). `mean` divides by the count of non-missing values.

**Q8. When should you drop a column instead of filling?**
When most of it is missing (for example over 50%) and it is not important, since imputed values would be mostly invented.

**Q9. How do you fill missing values in time series?**
`ffill`, `bfill`, or `interpolate` (linear or `method="time"`) after sorting by time.

**Q10. Why can filling with the mean hurt a model?**
It shrinks the variance, weakens relationships with the target, and ignores group differences. A group-wise or model-based fill preserves more structure.

---
# 20. Practice questions

Run this cell first to get the practice data.

In [55]:
houses = pd.DataFrame({
    "area":       [850, 1200, np.nan, 700, 2000, 1100, np.nan, 1500, 900, 1300, 1700, np.nan],
    "bedrooms":   [2, 3, 3, np.nan, 4, 2, 3, np.nan, 2, 3, 4, 3],
    "age_years":  [10, 5, np.nan, 15, 2, 8, np.nan, 12, np.nan, 6, 3, 9],
    "city":       ["Indore", "Bhopal", "Indore", None, "Pune", "Bhopal", "Indore", "Pune", None, "Bhopal", "Pune", "Indore"],
    "price_lakh": [45, 70, 85, 30, 120, 60, 55, np.nan, 48, 75, 105, 62],
    "floors":     ["1", "2", "?", "1", "3", "?", "2", "2", "1", "2", "3", "1"],
})
houses

,area,bedrooms,age_years,city,price_lakh,floors
0,850.0,2.0,10.0,Indore,45.0,1
1,1200.0,3.0,5.0,Bhopal,70.0,2
2,NaN,3.0,NaN,Indore,85.0,?
3,700.0,NaN,15.0,NaN,30.0,1
4,2000.0,4.0,2.0,Pune,120.0,3
5,1100.0,2.0,8.0,Bhopal,60.0,?
6,NaN,3.0,NaN,Indore,55.0,2
7,1500.0,NaN,12.0,Pune,NaN,2
8,900.0,2.0,NaN,NaN,48.0,1
9,1300.0,3.0,6.0,Bhopal,75.0,2


**Q1.** Show the missing count and missing percentage for every column (only columns that have missing values).
**Q2.** Find the rows that have **more than one** missing value.
**Q3.** Drop the rows where the target `price_lakh` is missing.
**Q4.** Fill `bedrooms` with its mode.
**Q5.** Fill `area` with the median of its own city, and anything still missing with the overall median.
**Q6.** Add an indicator column `age_missing`, then fill `age_years` with the median.
**Q7.** Fill missing `city` with `"Unknown"`.
**Q8.** The `floors` column has `"?"`. Convert it into real numbers (with NaN where it was `?`).
**Q9.** Fix the daily series below with interpolation, and also with `ffill`.
```python
s = pd.Series([100, np.nan, np.nan, 130, np.nan, 150], index=pd.date_range("2026-03-01", periods=6))
```
**Q10.** Drop every column that has more than 20% missing values.
**Q11.** Using only the **first 8 rows as train**, learn the median of `age_years` and use it to fill both the train part and the remaining rows (test).
**Q12.** `houses[houses["area"] > 1000]` leaves out the rows where `area` is missing. Write a filter that keeps rows with `area > 1000` **or** missing area.

### Solutions

In [56]:
# Q1
rep = pd.DataFrame({"missing": houses.isna().sum(), "missing_%": (houses.isna().mean() * 100).round(1)})
rep[rep["missing"] > 0]

,missing,missing_%
area,3,25.0
bedrooms,2,16.7
age_years,3,25.0
city,2,16.7
price_lakh,1,8.3


In [57]:
# Q2
houses[houses.isna().sum(axis=1) > 1]

,area,bedrooms,age_years,city,price_lakh,floors
2,NaN,3.0,NaN,Indore,85.0,?
3,700.0,NaN,15.0,NaN,30.0,1
6,NaN,3.0,NaN,Indore,55.0,2
7,1500.0,NaN,12.0,Pune,NaN,2
8,900.0,2.0,NaN,NaN,48.0,1


In [58]:
# Q3
h = houses.dropna(subset=["price_lakh"])
print(houses.shape, "->", h.shape)

(12, 6) -> (11, 6)


In [59]:
# Q4
h = houses.copy()
h["bedrooms"] = h["bedrooms"].fillna(h["bedrooms"].mode()[0])
print(h["bedrooms"].tolist())

[2.0, 3.0, 3.0, 3.0, 4.0, 2.0, 3.0, 3.0, 2.0, 3.0, 4.0, 3.0]


In [60]:
# Q5
h = houses.copy()
h["area"] = h["area"].fillna(h.groupby("city")["area"].transform("median"))
h["area"] = h["area"].fillna(h["area"].median())
print(h["area"].tolist())

[850.0, 1200.0, 850.0, 700.0, 2000.0, 1100.0, 850.0, 1500.0, 900.0, 1300.0, 1700.0, 850.0]


In [61]:
# Q6
h = houses.copy()
h["age_missing"] = h["age_years"].isna().astype(int)
h["age_years"] = h["age_years"].fillna(h["age_years"].median())
h[["age_years", "age_missing"]]

,age_years,age_missing
0,10.0,0
1,5.0,0
2,8.0,1
3,15.0,0
4,2.0,0
5,8.0,0
6,8.0,1
7,12.0,0
8,8.0,1
9,6.0,0


In [62]:
# Q7
houses["city"].fillna("Unknown").tolist()

['Indore',
 'Bhopal',
 'Indore',
 'Unknown',
 'Pune',
 'Bhopal',
 'Indore',
 'Pune',
 'Unknown',
 'Bhopal',
 'Pune',
 'Indore']

In [63]:
# Q8
floors = pd.to_numeric(houses["floors"].replace("?", np.nan))
print(floors.tolist())
print(floors.dtype)

[1.0, 2.0, nan, 1.0, 3.0, nan, 2.0, 2.0, 1.0, 2.0, 3.0, 1.0]
float64


In [64]:
# Q9
s = pd.Series([100, np.nan, np.nan, 130, np.nan, 150], index=pd.date_range("2026-03-01", periods=6))
print("interpolate:", s.interpolate().tolist())
print("ffill      :", s.ffill().tolist())

interpolate: [100.0, 110.0, 120.0, 130.0, 140.0, 150.0]
ffill      : [100.0, 100.0, 100.0, 130.0, 130.0, 150.0]


In [65]:
# Q10
keep = houses.columns[houses.isna().mean() <= 0.20]
print(keep.tolist())
houses[keep].head(3)

['bedrooms', 'city', 'price_lakh', 'floors']


,bedrooms,city,price_lakh,floors
0,2.0,Indore,45.0,1
1,3.0,Bhopal,70.0,2
2,3.0,Indore,85.0,?


In [66]:
# Q11
train, test = houses.iloc[:8].copy(), houses.iloc[8:].copy()
med = train["age_years"].median()
print("median from train:", med)

train["age_years"] = train["age_years"].fillna(med)
test["age_years"] = test["age_years"].fillna(med)
print(train["age_years"].isna().sum(), test["age_years"].isna().sum())

median from train: 9.0
0 0


In [67]:
# Q12
houses[(houses["area"] > 1000) | houses["area"].isna()]

,area,bedrooms,age_years,city,price_lakh,floors
1,1200.0,3.0,5.0,Bhopal,70.0,2
2,NaN,3.0,NaN,Indore,85.0,?
4,2000.0,4.0,2.0,Pune,120.0,3
5,1100.0,2.0,8.0,Bhopal,60.0,?
6,NaN,3.0,NaN,Indore,55.0,2
7,1500.0,NaN,12.0,Pune,NaN,2
9,1300.0,3.0,6.0,Bhopal,75.0,2
10,1700.0,4.0,3.0,Pune,105.0,3
11,NaN,3.0,9.0,Indore,62.0,1


---
# 21. Cheat sheet

```python
# DETECT
df.isna()   df.isna().sum()   (df.isna().mean() * 100).round(1)
df.isna().any(axis=1)   df[df.isna().any(axis=1)]
df.replace(["?", "N/A", -999], np.nan)           # fake missing -> NaN
pd.read_csv(path, na_values=["?", "-"])

# DROP
df.dropna()   df.dropna(subset=["a"])   df.dropna(thresh=3)   df.dropna(how="all")
df.dropna(axis=1)                                 # drop columns
df.loc[:, df.isna().mean() < 0.4]                 # keep columns with < 40% missing

# FILL
df["a"].fillna(0)                                 df.fillna({"a": 0, "b": "Unknown"})
df["a"].fillna(df["a"].mean())                    df["a"].fillna(df["a"].median())
df["c"].fillna(df["c"].mode()[0])
df["a"].fillna(df.groupby("g")["a"].transform("median"))
df["a"].fillna(df["b"])                           # from another column
s.ffill()   s.bfill()   s.interpolate()   s.interpolate(method="time")

# INDICATOR
df["a_missing"] = df["a"].isna().astype(int)

# NO LEAKAGE
value = train["a"].median()
train["a"] = train["a"].fillna(value);  test["a"] = test["a"].fillna(value)

# CHECK
df.isna().sum().sum()    # should be 0 before modelling
```

**Next chapter:** `08_data_cleaning`: duplicates, wrong data types, inconsistent text (`strip`, `lower`, `replace`, regex), outliers handling, and standardizing messy columns.